In [1]:
import os
import sys
import pandas as pd
import yaml 
from matplotlib import pyplot as plt
from matplotlib import ticker as mticker
import statsmodels.api as sm
import numpy as np
from itertools import product
import subprocess

with open("../../config.yaml.local", "r") as f:
    LOCAL_CONFIG = yaml.safe_load(f)
with open("../../config.yaml", "r") as f:
    CONFIG = yaml.safe_load(f)
sys.path.append("../python")

import globals
import data_tools as dt
import writing_tools as wt
import utils

LOCAL_PATH = LOCAL_CONFIG["LOCAL_PATH"]
RAW_DATA_PATH = LOCAL_CONFIG["RAW_DATA_PATH"]
DATA_PATH = LOCAL_CONFIG["DATA_PATH"]
R_PATH = LOCAL_CONFIG["R_PATH"]

RUN_R_SCRIPTS = False

RESULTS = {}

In [18]:
qual_df = pd.read_parquet(os.path.join(DATA_PATH, "post_quality_analysis_data.parquet"))
quant_df = pd.read_parquet(os.path.join(DATA_PATH, "post_quantity_analysis_data.parquet"))
territory_df = dt.get_territory_by_day_panel()  # territory-by-day


In [ ]:
# calculate days until next positive/negative posting fee change
# and days since last positive/negative posting fee change
# by territories, daily

territory_df["fee_diff"] = territory_df.groupby("subName")["posting_fee"].diff()

# days since
n_neg_changes = 0
n_pos_changes = 0
days_since_neg_change = np.nan
days_since_pos_change = np.nan
territory_df = territory_df.sort_values(["subName", "date"], ascending=True)
territory_df["days_since_pos_change"] = np.nan
territory_df["days_since_neg_change"] = np.nan
curr_sub = None
for idx, row in territory_df.iterrows():
    if row["subName"] != curr_sub:
        n_neg_changes = 0
        n_pos_changes = 0
        days_since_neg_change = np.nan
        days_since_pos_change = np.nan
        curr_sub = row["subName"]
    if row["fee_diff"] < 0:
        n_neg_changes += 1
        days_since_neg_change = 0
    elif row["fee_diff"] > 0:
        n_pos_changes += 1
        days_since_pos_change = 0
    
    territory_df.at[idx, "days_since_neg_change"] = days_since_neg_change
    territory_df.at[idx, "days_since_pos_change"] = days_since_pos_change
    
    if not np.isnan(days_since_neg_change):
        days_since_neg_change += 1
    if not np.isnan(days_since_pos_change):
        days_since_pos_change += 1

# days until 
n_neg_changes = 0
n_pos_changes = 0
days_until_neg_change = np.nan
days_until_pos_change = np.nan
territory_df = territory_df.sort_values(["subName", "date"], ascending=False)
territory_df["days_until_pos_change"] = np.nan
territory_df["days_until_neg_change"] = np.nan
curr_sub = None
for idx, row in territory_df.iterrows():
    if row["subName"] != curr_sub:
        n_neg_changes = 0
        n_pos_changes = 0
        days_until_neg_change = np.nan
        days_until_pos_change = np.nan
        curr_sub = row["subName"]
    if row["fee_diff"] < 0:
        n_neg_changes += 1
        days_until_neg_change = 0
    elif row["fee_diff"] > 0:
        n_pos_changes += 1
        days_until_pos_change = 0
    
    territory_df.at[idx, "days_until_neg_change"] = days_until_neg_change
    territory_df.at[idx, "days_until_pos_change"] = days_until_pos_change
    
    if not np.isnan(days_until_neg_change):
        days_until_neg_change += 1
    if not np.isnan(days_until_pos_change):
        days_until_pos_change += 1

territory_df = territory_df.sort_values(["subName", "date"], ascending=True)

territory_df["weeks_since_pos_change"] = np.floor(territory_df["days_since_pos_change"] / 7)
territory_df["weeks_since_neg_change"] = np.floor(territory_df["days_since_neg_change"] / 7)
territory_df["weeks_until_pos_change"] = np.ceil(territory_df["days_until_pos_change"] / 7)
territory_df["weeks_until_neg_change"] = np.ceil(territory_df["days_until_neg_change"] / 7)


In [31]:
qual_df["date"] = dt.as_date(qual_df["created_at"])

qual_df = qual_df.merge(
    territory_df[["subName", "date", "weeks_since_pos_change", "weeks_since_neg_change", "weeks_until_pos_change", "weeks_until_neg_change"]],
    how="left",
    on=["subName", "date"]
)




In [ ]:
qual_df.loc[(qual_df["weeks_since_pos_change"]<=12) | (qual_df["weeks_since_neg_change"]<=12) | (qual_df["weeks_until_pos_change"]<=12) | (qual_df["weeks_until_neg_change"]<=12)].shape

(85259, 29)